# DX 704 Week 7 Project

This week's project will investigate issues in a quadcopter controller based using a linear quadratic regulator.
You will start with an existing model of the system and logs from a quadcopter based on it, investigate discrepancies, and ultimately train a new model of the system dynamics.

The full project description and a template notebook are available on GitHub: [Project 7 Materials](https://github.com/bu-cds-dx704/dx704-project-07).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Introduction

You've just joined a drone startup.
On your first day, you join your new team to watch a test flight for a new quadcopter prototype.
Watching the prototype fly, the team comments that it is not as smooth as usual and suspects that something is off in the controller.
They provide you a copy of the dynamics model and log data from the prototype to investigate.

The quadcopter control model is a slightly more sophisticated version of the 1D quadcopter problem previously considered.

The state vector $\mathbf{x}_t$ now includes an acceleration component, and the action vector now has a servo control for the throttle instead of a raw force component.
\begin{array}{rcl}
\mathbf{x}_t & = & \begin{bmatrix} x_t \\ v_t \\ a_t \end{bmatrix} \\
\mathbf{u_t} & = & \begin{bmatrix} u_t \end{bmatrix}
\end{array}

## Part 1: Reconstruct the Control Matrix

You are provided the dynamics model in the files `model-A.tsv`, `model-B.tsv`, `cost-Q.tsv` and `cost-R.tsv`.
Recompute the control matrix $\mathbf{K}$ to be used in the infinite horizon linear quadratic regulator problem.

In [1]:
import numpy as np
import pandas as pd
from scipy.linalg import solve_discrete_are

A = pd.read_csv('model-A.tsv', sep='\t', index_col=0).to_numpy(dtype=float)
B = pd.read_csv('model-B.tsv', sep='\t', index_col=0).to_numpy(dtype=float)
Q = pd.read_csv('cost-Q.tsv', sep='\t', index_col=0).to_numpy(dtype=float)
R = pd.read_csv('cost-R.tsv', sep='\t', index_col=0).to_numpy(dtype=float)

P = solve_discrete_are(A, B, Q, R)
control_K = np.linalg.solve(R + B.T @ P @ B, B.T @ P @ A)

assert control_K.shape == (1, 3)
assert np.isfinite(control_K).all()

control_K_df = pd.DataFrame(control_K, columns=['x', 'v', 'a'])
control_K_df

,x,v,a
0,0.33446,1.304454,1.858131


Save $\mathbf{K}$ in a file "control-K-intended.tsv" with columns x, v and a.

In [2]:
control_K_df.to_csv(
    'control-K-intended.tsv',
    sep='\t',
    index=False
)
control_K_df

,x,v,a
0,0.33446,1.304454,1.858131


Submit "control-K-intended.tsv" in Gradescope.

## Part 2: Recompute the Actions for the Logged States

You get access to the log data for the prototype as the file "qc-log.tsv".
It conveniently saves all the state and actions made.
Recompute the actions based on your $\mathbf{K}$ matrix computed in part 1.

In [3]:
import numpy as np
import pandas as pd

qc_log = pd.read_csv('qc-log.tsv', sep='\t')
control_K_df = pd.read_csv('control-K-intended.tsv', sep='\t')

state_values = qc_log[['x', 'v', 'a']].to_numpy(dtype=float)
control_coefficients = control_K_df[['x', 'v', 'a']].to_numpy(dtype=float).reshape(-1)
u_check = -(state_values @ control_coefficients)

qc_check = pd.DataFrame({
    'index': qc_log['index'],
    'u_check': u_check
})

assert len(qc_check) == len(qc_log)
assert np.isfinite(qc_check['u_check']).all()
qc_check.head()

,index,u_check
0,0,1.672299
1,1,-1.152095
2,2,-1.235183
3,3,-0.288504
4,4,0.369202


Save your computed actions as "qc-check.tsv" with columns "index" and "u_check".

In [4]:
qc_check.to_csv('qc-check.tsv', sep='\t', index=False)
qc_check

,index,u_check
0,0,1.672299e+00
1,1,-1.152095e+00
2,2,-1.235183e+00
3,3,-2.885035e-01
4,4,3.692016e-01
...,...,...
95,95,-5.148677e-19
96,96,2.552224e-19
97,97,3.971337e-19
98,98,1.886533e-19


Submit "qc-check.tsv" in Gradescope.

## Part 3: Reverse Engineer the Actual Control Matrix

Now that you have found a systematic difference between your computed actions and the logged actions, estimate $
$, the control matrix that was actually used to choose actions in the prototype.

Hint: With a linear quadratic regulator, the optimal actions are always linear combinations of the state that are calculated using the control matrix.
You can use linear regression to reverse-engineer the coefficients in the control matrix.

In [5]:
qc_log = pd.read_csv('qc-log.tsv', sep='\t')

logged_states = qc_log[['x', 'v', 'a']].to_numpy(dtype=float)
logged_actions = qc_log['u'].to_numpy(dtype=float)
regression_coefficients, residuals, rank, singular_values = np.linalg.lstsq(
    logged_states,
    logged_actions,
    rcond=None
)

control_K_actual = -regression_coefficients.reshape(1, -1)
control_K_actual_df = pd.DataFrame(
    control_K_actual,
    columns=['x', 'v', 'a']
)

predicted_actions = logged_states @ regression_coefficients
rmse = np.sqrt(np.mean((logged_actions - predicted_actions) ** 2))
assert control_K_actual.shape == (1, 3)
assert np.isfinite(control_K_actual).all()

print(f'Regression rank: {rank}')
print(f'Action reconstruction RMSE: {rmse:.8f}')
control_K_actual_df

Regression rank: 3
Action reconstruction RMSE: 0.00000000


,x,v,a
0,0.340438,1.30012,1.950117


Save $\mathbf{K}_{\mathrm{actual}}$ in "control-K-actual.tsv" with the same format as "control-K-intended.tsv".

In [6]:
control_K_actual_df.to_csv(
    'control-K-actual.tsv',
    sep='\t',
    index=False
)
control_K_actual_df

,x,v,a
0,0.340438,1.30012,1.950117


Submit "control-k-actual.tsv" in Gradescope.

## Part 4: Recompute the System Dynamics from the Log Data

On further investigation, it turns out that the control matrix $\mathbf{K}$ in the prototype was modified to compensate for state prediction errors.
You would like to recompute the $\mathbf{A}$ and $\mathbf{B}$ matrices using the log data since they are used to predict the next states.
However, since the action vector $\mathbf{u}_t$ is linearly dependent on the state via $\mathbf{u}_t=-\mathbf{K} \mathbf{x}_t$, you need a new data set so you can separate the effects of the $\mathbf{A}$ and $\mathbf{B}$ matrices.
Your co-workers kindly provide a new file "qc-train.tsv" where noise was added to each action.
Estimate the true $\mathbf{A}$ and $\mathbf{B}$ matrices based on this file.

In [7]:
import numpy as np
import pandas as pd

qc_train = pd.read_csv('qc-train.tsv', sep='\t')
state_columns = ['x', 'v', 'a']
current_states = qc_train[state_columns].to_numpy(dtype=float)
actions = qc_train['u'].to_numpy(dtype=float)

# The next row records the state reached after each row's action.
design = np.column_stack((current_states[:-1], actions[:-1]))
next_states = current_states[1:]
coefficients, residuals, rank, singular_values = np.linalg.lstsq(
    design,
    next_states,
    rcond=None
)

A_estimated = coefficients[:3, :].T
B_estimated = coefficients[3, :].reshape(3, 1)
predicted_next_states = design @ coefficients
rmse = np.sqrt(np.mean((next_states - predicted_next_states) ** 2))

assert rank == 4
assert np.isfinite(A_estimated).all()
assert np.isfinite(B_estimated).all()

A_new_df = pd.DataFrame(
    A_estimated,
    index=state_columns,
    columns=state_columns
)
B_new_df = pd.DataFrame(
    B_estimated,
    index=state_columns,
    columns=['u']
)

print(f'Regression rank: {rank}')
print(f'One-step state prediction RMSE: {rmse:.8f}')
print('Estimated A:')
display(A_new_df)
print('Estimated B:')
display(B_new_df)

Regression rank: 4
One-step state prediction RMSE: 0.00000000
Estimated A:


,x,v,a
x,1.000000e+00,1.100000e+00,6.096590e-16
v,1.251418e-16,9.000000e-01,9.500000e-01
a,4.812943e-17,1.026956e-15,1.100000e+00


Estimated B:


,u
x,7.250324e-17
v,-1.000000e-02
a,9.000000e-01


Save $\mathbf{A}$ and $\mathbf{B}$ to "model-A-new.tsv" and "model-B-new.tsv" respectively.

In [8]:
A_new_df.to_csv(
    'model-A-new.tsv',
    sep='\t',
    index=True,
    index_label='index'
)
B_new_df.to_csv(
    'model-B-new.tsv',
    sep='\t',
    index=True,
    index_label='index'
)
print('Estimated A:')
display(A_new_df)
print('Estimated B:')
display(B_new_df)

Estimated A:


,x,v,a
x,1.000000e+00,1.100000e+00,6.096590e-16
v,1.251418e-16,9.000000e-01,9.500000e-01
a,4.812943e-17,1.026956e-15,1.100000e+00


Estimated B:


,u
x,7.250324e-17
v,-1.000000e-02
a,9.000000e-01


Submit "model-A-new.tsv" and "model-B-new.tsv" in Gradescope.

## Part 5: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 6: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.

### Acknowledgement

No other people provided assistance. I used NumPy for matrix calculations, pandas for reading and writing TSV data, and SciPy's `solve_discrete_are` to solve the discrete-time algebraic Riccati equation. GitHub Copilot Chat in Visual Studio Code assisted with interpreting the assignment, writing and debugging the calculations, and validating the outputs.

[Copilot Chat transcript](file:///Users/christopherabbas/Library/Application%20Support/Code/User/workspaceStorage/d690c5667068afdbf94ff162bc48cae2/GitHub.copilot-chat/transcripts/e7d90d9c-3572-4733-8668-e31c0026bd5e.jsonl)